#### 01. Find the total matches played, total wins & total losses by each team

In [0]:
%sql
---- Creating new catalog, schema -----
use catalog sql_youtube_practise;
create schema if not exists pyspark;
use pyspark;
show current schema;

catalog,namespace
sql_youtube_practise,pyspark


In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
data = [
    ("India", "SL", "India"),
    ("SL", "Aus", "Aus"),
    ("SA", "Eng", "Eng"),
    ("Eng", "NZ", "NZ"),
    ("Aus", "India", "India")
]
columns = ["Team_1", "Team_2", "Winner"]
icc_world_cup_df = spark.createDataFrame(data, columns)
icc_world_cup_df.write.mode("overwrite").saveAsTable("icc_world_cup")
icc_world_cup_df.display()

Team_1,Team_2,Winner
India,SL,India
SL,Aus,Aus
SA,Eng,Eng
Eng,NZ,NZ
Aus,India,India


In [0]:
from pyspark.sql.functions import *
team1_df = icc_world_cup_df.select(
    col("Team_1").alias("team_name"),
    when(col("Winner") == col("Team_1"), 1).otherwise(0).alias("wins")
)
team2_df = icc_world_cup_df.select(
    col("Team_2").alias("team_name"),
    when(col("Winner") == col("Team_2"), 1).otherwise(0).alias("wins")
)
team = team1_df.unionAll(team2_df) \
        .groupBy("team_name") \
        .agg(
            count("*").alias("total_matches_played"),
            sum("Wins").alias("Wins")
        ) \
        .withColumn(
            "Losses",
            col("total_matches_played") - col("Wins")
        ) \
        .orderBy(col("Wins").desc())
team.display() 

team_name,total_matches_played,Wins,Losses
India,2,2,0
Eng,2,1,1
Aus,2,1,1
NZ,1,1,0
SL,2,0,2
SA,1,0,1


#### 02. Find the number of first-time customers and repeat customers for each order date.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
spark = SparkSession.builder.getOrCreate()
data = [
    (1, 100, "2022-01-01", 2000),
    (2, 200, "2022-01-01", 2500),
    (3, 300, "2022-01-01", 2100),
    (4, 100, "2022-01-02", 2000),
    (5, 400, "2022-01-02", 2200),
    (6, 500, "2022-01-02", 2700),
    (7, 100, "2022-01-03", 3000),
    (8, 400, "2022-01-03", 1000),
    (9, 600, "2022-01-03", 3000)
]
columns = ["order_id", "customer_id", "order_date", "order_amount"]
customer_orders_df = (
    spark.createDataFrame(data, columns)
         .withColumn("order_date", col("order_date").cast("date"))
)
customer_orders_df.write.mode("overwrite").saveAsTable("customer_orders")
customer_orders_df.display()

order_id,customer_id,order_date,order_amount
1,100,2022-01-01,2000
2,200,2022-01-01,2500
3,300,2022-01-01,2100
4,100,2022-01-02,2000
5,400,2022-01-02,2200
6,500,2022-01-02,2700
7,100,2022-01-03,3000
8,400,2022-01-03,1000
9,600,2022-01-03,3000


In [0]:
from pyspark.sql.functions import col, min, when, sum

# Step 1: First visit date of each customer
customer_first_visit = (
    customer_orders_df
    .groupBy("customer_id")
    .agg(
        min("order_date").alias("first_visit_date")
    )
)
customer_first_visit.display()

# Step 2: Join and create flags
customer_orders_new_df = (
    customer_orders_df
    .join(customer_first_visit, on="customer_id", how="inner")
    .withColumn(
        "first_visit_flag",
        when(col("order_date") == col("first_visit_date"), 1).otherwise(0)
    )
    .withColumn(
        "repeat_visit_flag",
        when(col("order_date") != col("first_visit_date"), 1).otherwise(0)
    )
)
customer_orders_new_df.display()

# Step 3: Aggregate
final_df = (
    customer_orders_new_df
    .groupBy("order_date")
    .agg(
        sum("first_visit_flag").alias("first_visits"),
        sum("repeat_visit_flag").alias("repeat_visits")
    )
    .orderBy("order_date")
)
final_df.display()

customer_id,first_visit_date
100,2022-01-01
200,2022-01-01
300,2022-01-01
400,2022-01-02
500,2022-01-02
600,2022-01-03


customer_id,order_id,order_date,order_amount,first_visit_date,first_visit_flag,repeat_visit_flag
100,7,2022-01-03,3000,2022-01-01,0,1
200,2,2022-01-01,2500,2022-01-01,1,0
300,3,2022-01-01,2100,2022-01-01,1,0
400,8,2022-01-03,1000,2022-01-02,0,1
500,6,2022-01-02,2700,2022-01-02,1,0
600,9,2022-01-03,3000,2022-01-03,1,0
100,4,2022-01-02,2000,2022-01-01,0,1
400,5,2022-01-02,2200,2022-01-02,1,0
100,1,2022-01-01,2000,2022-01-01,1,0


order_date,first_visits,repeat_visits
2022-01-01,3,0
2022-01-02,2,1
2022-01-03,1,2


###### Using windows fucntion

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

window_spec = Window.partitionBy("customer_id")

final_df = (
    customer_orders_df
    .withColumn(
        "first_visit_date",
        min("order_date").over(window_spec)
    )
    .withColumn(
        "first_visit_flag",
        when(col("order_date") == col("first_visit_date"), 1).otherwise(0)
    )
    .withColumn(
        "repeat_visit_flag",
        when(col("order_date") != col("first_visit_date"), 1).otherwise(0)
    )
    .groupBy("order_date")
    .agg(
        sum("first_visit_flag").alias("first_visits"),
        sum("repeat_visit_flag").alias("repeat_visits")
    )
    .orderBy("order_date")
)
final_df.display()

order_date,first_visits,repeat_visits
2022-01-01,3,0
2022-01-02,2,1
2022-01-03,1,2


#### 03. Find each employee's 
- most visited floor, 
- total number of visits, and
- distinct resources they used.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
spark = SparkSession.builder.getOrCreate()
data = [
    ("A", "Bangalore", "A@gmail.com", 1, "CPU"),
    ("A", "Bangalore", "A1@gmail.com", 1, "CPU"),
    ("A", "Bangalore", "A2@gmail.com", 2, "DESKTOP"),
    ("B", "Bangalore", "B@gmail.com", 2, "DESKTOP"),
    ("B", "Bangalore", "B1@gmail.com", 2, "DESKTOP"),
    ("B", "Bangalore", "B2@gmail.com", 1, "MONITOR")
]
columns = ["name", "address", "email", "floor", "resources"]
entries_df = spark.createDataFrame(data, columns)
entries_df.write.mode("overwrite").saveAsTable("entries")
entries_df.display()

name,address,email,floor,resources
A,Bangalore,A@gmail.com,1,CPU
A,Bangalore,A1@gmail.com,1,CPU
A,Bangalore,A2@gmail.com,2,DESKTOP
B,Bangalore,B@gmail.com,2,DESKTOP
B,Bangalore,B1@gmail.com,2,DESKTOP
B,Bangalore,B2@gmail.com,1,MONITOR


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

part1 = entries_df.groupBy(
    "name"
    ).agg(
        count(col("floor")).alias("total_visits"),
        concat_ws(",", collect_set(col("resources"))).alias("resources_used")
    )
part1.display()

part2 = entries_df.groupBy(
    col("name"),col("floor")
).agg(
    count(col("floor")).alias("no_of_floor_visit")
).withColumn(
    "rank",
    dense_rank().over(Window.partitionBy(col("name")).orderBy(col("no_of_floor_visit").desc()))
)
part2.display()

final_df = part2.join(
    part1,
    part2["name"] == part1["name"],
    "inner"
).filter(
    col("rank") == 1
).select(
    part1["name"],
    part1["total_visits"],
    part2["floor"].alias("most_visited_floor"),
    part1["resources_used"]
).orderBy(col("name"))
final_df.display()

name,total_visits,resources_used
A,3,"CPU,DESKTOP"
B,3,"DESKTOP,MONITOR"


name,floor,no_of_floor_visit,rank
A,1,2,1
A,2,1,2
B,2,2,1
B,1,1,2


name,total_visits,most_visited_floor,resources_used
A,3,1,"CPU,DESKTOP"
B,3,2,"DESKTOP,MONITOR"


##### 04. Find the person IDs and names of all users whose friends' total score is greater than 100, along with the number of friends and their total score.

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
person_data = [
    (1, "Alice", "alice2018@hotmail.com", 88),
    (2, "Bob", "bob2018@hotmail.com", 11),
    (3, "Davis", "davis2018@hotmail.com", 27),
    (4, "Tara", "tara2018@hotmail.com", 45),
    (5, "John", "john2018@hotmail.com", 63)
]
person_columns = ["PersonID", "Name", "Email", "Score"]
person_df = spark.createDataFrame(person_data, person_columns)
person_df.write.mode("Overwrite").saveAsTable("person")

friend_data = [
    (1, 2),
    (1, 3),
    (2, 1),
    (2, 3),
    (3, 5),
    (4, 2),
    (4, 3),
    (4, 5)
]
friend_columns = ["PersonID", "FriendID"]
friend_df = spark.createDataFrame(friend_data, friend_columns)
friend_df.write.mode("Overwrite").saveAsTable("friend")

friend_df.display()
person_df.display()

PersonID,FriendID
1,2
1,3
2,1
2,3
3,5
4,2
4,3
4,5


PersonID,Name,Email,Score
1,Alice,alice2018@hotmail.com,88
2,Bob,bob2018@hotmail.com,11
3,Davis,davis2018@hotmail.com,27
4,Tara,tara2018@hotmail.com,45
5,John,john2018@hotmail.com,63


In [0]:
from pyspark.sql.functions import col, asc, count, sum

a = friend_df.alias("f").join(
    person_df.alias("p"),
    col("f.FriendID") == col("p.PersonID"),
    "inner"
).groupBy(
    col("f.PersonID")
).agg(
    count(col("f.FriendID")).alias("total_friends"),
    sum(col("p.Score")).alias("total_score")
).filter(
    col("total_score") > 100
)

final_df = a.join(
    person_df.alias("p1"),
    a["PersonID"] == col("p1.PersonID"),
    "inner"
).select(
    a["PersonID"],
    col("p1.Name"),
    a["total_friends"],
    a["total_score"]
)

a.display()
final_df.display()

PersonID,total_friends,total_score
2,2,115
4,3,101


PersonID,Name,total_friends,total_score
2,Bob,2,115
4,Tara,3,101


##### 05. Calculate the daily cancellation percentage for trips where both the client and the driver are not banned.

###### Concept: Filter valid trips, count cancelled and total trips for each day, then calculate the cancellation percentage.

Formula:
Cancellation Rate (%) = (Cancelled Trips / Total Trips) × 100

In [0]:
from pyspark.sql.types import *
from datetime import date

# ----------------------------
# Trips DataFrame
# ----------------------------

trips_data = [
    (1, 1, 10, 1, "completed", date(2013, 10, 1)),
    (2, 2, 11, 1, "cancelled_by_driver", date(2013, 10, 1)),
    (3, 3, 12, 6, "completed", date(2013, 10, 1)),
    (4, 4, 13, 6, "cancelled_by_client", date(2013, 10, 1)),
    (5, 1, 10, 1, "completed", date(2013, 10, 2)),
    (6, 2, 11, 6, "completed", date(2013, 10, 2)),
    (7, 3, 12, 6, "completed", date(2013, 10, 2)),
    (8, 2, 12, 12, "completed", date(2013, 10, 3)),
    (9, 3, 10, 12, "completed", date(2013, 10, 3)),
    (10, 4, 13, 12, "cancelled_by_driver", date(2013, 10, 3))
]

trips_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("client_id", IntegerType(), True),
    StructField("driver_id", IntegerType(), True),
    StructField("city_id", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("request_at", DateType(), True)
])

trips_df = spark.createDataFrame(
    trips_data,
    schema=trips_schema
)

trips_df.display()
trips_df.write.mode("overwrite").saveAsTable("trips")

# ----------------------------
# Users DataFrame
# ----------------------------

users_data = [
    (1, "No", "client"),
    (2, "Yes", "client"),
    (3, "No", "client"),
    (4, "No", "client"),
    (10, "No", "driver"),
    (11, "No", "driver"),
    (12, "No", "driver"),
    (13, "No", "driver")
]

users_schema = StructType([
    StructField("users_id", IntegerType(), True),
    StructField("banned", StringType(), True),
    StructField("role", StringType(), True)
])

users_df = spark.createDataFrame(
    users_data,
    schema=users_schema
)

users_df.display()
users_df.write.mode("overwrite").saveAsTable("users")

id,client_id,driver_id,city_id,status,request_at
1,1,10,1,completed,2013-10-01
2,2,11,1,cancelled_by_driver,2013-10-01
3,3,12,6,completed,2013-10-01
4,4,13,6,cancelled_by_client,2013-10-01
5,1,10,1,completed,2013-10-02
6,2,11,6,completed,2013-10-02
7,3,12,6,completed,2013-10-02
8,2,12,12,completed,2013-10-03
9,3,10,12,completed,2013-10-03
10,4,13,12,cancelled_by_driver,2013-10-03


users_id,banned,role
1,No,client
2,Yes,client
3,No,client
4,No,client
10,No,driver
11,No,driver
12,No,driver
13,No,driver


In [0]:
from pyspark.sql.functions import *

trips = trips_df.alias("t").join(
    users_df.alias("u"),
    col("t.client_id") == col("u.users_id"),
    "inner"
).join(
    users_df.alias("u1"),
    col("t.driver_id") == col("u1.users_id"),
    "inner"
).filter(
    (col("u.banned") == "No") &
    (col("u1.banned") == "No")
).groupBy(
    col("t.request_at")
).agg(
    count(lit(1)).alias("total_trips"),
    count(
        when(col("t.status") != "completed", 1)
    ).alias("cancelled_trips")
)
final_df = trips.select(
    col("request_at"),
    col("cancelled_trips"),
    col("total_trips"),
    round(
        col("cancelled_trips") * 100 / col("total_trips"),
        2
    ).alias("cancellation_rate")
).orderBy(
    col("request_at")
)

final_df.display()

request_at,cancelled_trips,total_trips,cancellation_rate
2013-10-01,1,3,33.33
2013-10-02,0,2,0.0
2013-10-03,1,2,50.0


#### 06. Find the winner of each group based on the highest total score. If two players have the same score, choose the player with the smaller Player_ID to be considered.

Concept:
Each match contains scores for two players in different columns.
Convert them into one column using UNION ALL, calculate total score for each player, join with the Players table, rank players within each group, and
select Rank = 1.

In [0]:
from pyspark.sql.types import *

# ----------------------------
# Players DataFrame
# ----------------------------

players_data = [
    (15, 1),
    (25, 1),
    (30, 1),
    (45, 1),
    (10, 2),
    (35, 2),
    (50, 2),
    (20, 3),
    (40, 3)
]
players_schema = StructType([
    StructField("player_id", IntegerType(), True),
    StructField("group_id", IntegerType(), True)
])
players_df = spark.createDataFrame(
    players_data,
    players_schema
)
players_df.display()
players_df.write.mode("overwrite").saveAsTable("players")


# ----------------------------
# Matches DataFrame
# ----------------------------

matches_data = [
    (1, 15, 45, 3, 0),
    (2, 30, 25, 1, 2),
    (3, 30, 15, 2, 0),
    (4, 40, 20, 5, 2),
    (5, 35, 50, 1, 1)
]
matches_schema = StructType([
    StructField("match_id", IntegerType(), True),
    StructField("first_player", IntegerType(), True),
    StructField("second_player", IntegerType(), True),
    StructField("first_score", IntegerType(), True),
    StructField("second_score", IntegerType(), True)
])
matches_df = spark.createDataFrame(
    matches_data,
    matches_schema
)
matches_df.display()
matches_df.write.mode("overwrite").saveAsTable("matches")

player_id,group_id
15,1
25,1
30,1
45,1
10,2
35,2
50,2
20,3
40,3


match_id,first_player,second_player,first_score,second_score
1,15,45,3,0
2,30,25,1,2
3,30,15,2,0
4,40,20,5,2
5,35,50,1,1


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

players_scores = matches_df.select(
    col("first_player"),
    col("first_score")
).unionAll(
    matches_df.select(
        col("second_player"),
        col("second_score")
    )
).groupBy(
    col("first_player").alias("player")
).agg(
    sum(col("first_score")).alias("total_score")
)
players_scores.display()


players_rank = players_df.alias("p").join(
    players_scores.alias("s"),
    col("p.player_id") == col("s.player"),
    "left"
).select(
    col("p.player_id"),
    col("p.group_id"),
    coalesce(col("s.total_score"), lit(0)).alias("total_score")
).withColumn(
    "rank",
    dense_rank().over(
        Window.partitionBy(
            col("group_id")
        ).orderBy(
            desc("total_score"),
            asc("player_id")
        )
    )
)
players_rank.display()


final_df = players_rank.filter(
    col("rank") == 1
).select(
    col("group_id"),
    col("player_id"),
    col("total_score")
).orderBy(
    col("group_id")
)
final_df.display()

player,total_score
15,3
30,3
40,5
35,1
45,0
25,2
20,2
50,1


player_id,group_id,total_score,rank
15,1,3,1
30,1,3,2
25,1,2,3
45,1,0,4
35,2,1,1
50,2,1,2
10,2,0,3
40,3,5,1
20,3,2,2


group_id,player_id,total_score
1,15,3
2,35,1
3,40,5


####07. For every seller, determine whether the brand of the second item they sold matches their favorite brand.
Rank each seller's orders by sale date to identify the second sale.
Compare the brand of the second sold item with the seller's favorite brand.
Use a LEFT JOIN to include sellers who have sold fewer than two items.

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import to_date

# ------------------------------------------------------------------- #
########  Marketplace Users DataFrame   ###########
# ------------------------------------------------------------------- #
marketplace_users_data = [
    (1, "2019-01-01", "Lenovo"),
    (2, "2019-02-09", "Samsung"),
    (3, "2019-01-19", "LG"),
    (4, "2019-05-21", "HP")
]
marketplace_users_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("join_date", StringType(), True),
    StructField("favorite_brand", StringType(), True)
])
marketplace_users_df = (
    spark.createDataFrame(
        marketplace_users_data,
        marketplace_users_schema
    ).withColumn(
        "join_date",
        to_date("join_date")
    )
)
marketplace_users_df.display()
marketplace_users_df.write.mode("overwrite").saveAsTable("marketplace_users")

# ------------------------------------------------------------------- #
########  Marketplace Orders DataFrame   ###########
# ------------------------------------------------------------------- #
marketplace_orders_data = [
    (1, "2019-08-01", 4, 1, 2),
    (2, "2019-08-02", 2, 1, 3),
    (3, "2019-08-03", 3, 2, 3),
    (4, "2019-08-04", 1, 4, 2),
    (5, "2019-08-04", 1, 3, 4),
    (6, "2019-08-05", 2, 2, 4)
]
marketplace_orders_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("order_date", StringType(), True),
    StructField("item_id", IntegerType(), True),
    StructField("buyer_id", IntegerType(), True),
    StructField("seller_id", IntegerType(), True)
])
marketplace_orders_df = (
    spark.createDataFrame(
        marketplace_orders_data,
        marketplace_orders_schema
    ).withColumn(
        "order_date",
        to_date("order_date")
    )
)
marketplace_orders_df.display()
marketplace_orders_df.write.mode("overwrite").saveAsTable("marketplace_orders")

# ------------------------------------------------------------------- #
########  Marketplace Items DataFrame   ###########
# ------------------------------------------------------------------- #
marketplace_items_data = [
    (1, "Samsung"),
    (2, "Lenovo"),
    (3, "LG"),
    (4, "HP")
]
marketplace_items_schema = StructType([
    StructField("item_id", IntegerType(), True),
    StructField("item_brand", StringType(), True)
])
marketplace_items_df = spark.createDataFrame(
    marketplace_items_data,
    marketplace_items_schema
)
marketplace_items_df.display()
marketplace_items_df.write.mode("overwrite").saveAsTable("marketplace_items")

user_id,join_date,favorite_brand
1,2019-01-01,Lenovo
2,2019-02-09,Samsung
3,2019-01-19,LG
4,2019-05-21,HP


order_id,order_date,item_id,buyer_id,seller_id
1,2019-08-01,4,1,2
2,2019-08-02,2,1,3
3,2019-08-03,3,2,3
4,2019-08-04,1,4,2
5,2019-08-04,1,3,4
6,2019-08-05,2,2,4


item_id,item_brand
1,Samsung
2,Lenovo
3,LG
4,HP


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

cte = marketplace_orders_df.alias("o").join(
    marketplace_items_df.alias("i"),
    col("o.item_id") == col("i.item_id"),
    "inner"
).select(
    col("o.item_id"),
    col("o.seller_id"),
    col("o.order_date"),
    col("i.item_brand"),
    dense_rank().over(Window.partitionBy(col("o.seller_id")).orderBy(col("o.order_date"))).alias("rank")
)

final_df = marketplace_users_df.alias("u").join(
    cte.alias("c"),
    (col("u.user_id") == col("c.seller_id")) & (col("c.rank") == 2),
    "left"
).select(
    col("u.user_id").alias("seller_id"),
    col("c.order_date"),
    col("c.item_brand"),
    col("u.favorite_brand"),
    when(
        col("c.item_brand") == col("u.favorite_brand"),
        "Yes"
    ).otherwise("No").alias("is_favorite")
)
final_df.display()

seller_id,order_date,item_brand,favorite_brand,is_favorite
1,null,null,Lenovo,No
2,2019-08-04,Samsung,Samsung,Yes
3,2019-08-03,LG,LG,Yes
4,2019-08-05,Lenovo,HP,No


##### 08. Find the continuous date ranges for each task state (success/fail), showing the start date and end date of every consecutive sequence.

###### Identify consecutive records having the same state by assigning them to the same group, then return the start and end date for each continuous sequence.

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import to_date
tasks_data = [
    ("2019-01-01", "success"),
    ("2019-01-02", "success"),
    ("2019-01-03", "success"),
    ("2019-01-04", "fail"),
    ("2019-01-05", "fail"),
    ("2019-01-06", "success")
]
tasks_schema = StructType([
    StructField("date_value", StringType(), True),
    StructField("state", StringType(), True)
])
tasks_df = (
    spark.createDataFrame(
        tasks_data,
        tasks_schema
    ).withColumn(
        "date_value",
        to_date("date_value")
    )
)
tasks_df.display()
tasks_df.write.mode("overwrite").saveAsTable("tasks")

date_value,state
2019-01-01,success
2019-01-02,success
2019-01-03,success
2019-01-04,fail
2019-01-05,fail
2019-01-06,success


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

final_df = tasks_df.withColumn(
    "rank",
    row_number().over(Window.partitionBy(col("state")).orderBy(col("date_value")))
).withColumn(
    "group",
    date_sub(col("date_value"), (col("rank")))
).groupBy(
    col("group"), col("state")
).agg(
    max(col("date_value")).alias("end_date"),
    min(col("date_value")).alias("start_date"),
    col("state")
).select(
    col("start_date"),
    col("end_date"),
    col("state")
).orderBy(
    col("start_date")
)

final_df.display()

start_date,end_date,state
2019-01-01,2019-01-03,success
2019-01-04,2019-01-05,fail
2019-01-06,2019-01-06,success
